# 04 — Run the Trained Model Live

Load the current trained MOUSE model from the Hub and evaluate it on Procedural Frozen Lake.

Both training notebooks push to the same `MODEL_ID`. This notebook pulls whatever checkpoint is currently on the Hub, so if you run offline training and push, it evaluates that model; if you run online training and push afterward, it evaluates the online-trained model instead.

This notebook covers two things:
1. **Score per task** — each env runs `TASKS_PER_ENV` tasks; a task is a fresh held-out map with `EPISODES_PER_TASK` episodes to figure it out in context, and the score is the points earned in those episodes (0 to 5). Dying in a hole and timing out both cost exactly one episode of the task's budget, so neither is a shortcut.
2. **Replay animation** — capture a short episode and play it back as an inline animation.

In [ ]:
import os
import torch
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML, display

import procedural_frozenlake  # noqa: F401 — registers Procedural-FrozenLake-v1
from mouse_gym import EnvConfig, make_group_env
from mouse_core import load_model


# FrozenLake renders via pygame; run headless in environments without a display.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")
os.environ.setdefault("SDL_AUDIODRIVER", "dummy")


MODEL_ID = "mouse-example-model-offline"       # Hub repo id for load_model
TASKS_PER_ENV = 4                      # tasks per env; each task is a fresh map with EPISODES_PER_TASK episodes
EPISODES_PER_TASK = 20                  # episodes per task (must match the training notebooks)
MAX_EPISODE_STEPS = 30                 # episode truncation horizon (must match the training notebooks)
NUM_ENVS = 10                          # parallel env instances for evaluation
NUM_VIDEOS = 10                        # env instances that capture rgb_array frames for replay
MAX_VIDEO_FRAMES = 200                 # cap on captured frames per env so replays stay small


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Environment

`NUM_ENVS` independent env instances run in parallel. Each one is specified by its own `EnvConfig`, with `reset_seed=i` for its reset stream and `map_seed=i + 1000000` seeding a held-out map stream (training notebooks use `map_seed=i`, so this evaluates generalization to unseen layouts). Each env runs 5-episode tasks (`episodes_per_task=EPISODES_PER_TASK`) that regenerate the map — and its id relabeling — at every task boundary, with deterministic movement (`slippery_success_rate=1.0`) and `MAX_EPISODE_STEPS` (30-step) episode truncation, matching the training env config. `permute_obs` / `permute_actions` are enabled to match training too: each held-out map comes with its own unknown relabeling, so the model must infer both the layout and the labeling from context within the task. Only the first `NUM_VIDEOS` envs use `render_mode="rgb_array"`, so evaluation can cover more envs without embedding a video for every one.

In [2]:
configs = [
    EnvConfig(
        id="Procedural-FrozenLake-v1",
        name=f"proc_frozenlake_{i}",
        reset_seed=i,
        episodes_per_task=EPISODES_PER_TASK,
        task_reset_options={"regenerate_map": True},  # fresh map (and relabeling) every task
        kwargs={
            "width": 8,
            "height": 8,
            **({"render_mode": "rgb_array"} if i < NUM_VIDEOS else {}),
            "max_episode_steps": MAX_EPISODE_STEPS,
            "map_seed": i + 1000000,
            "slippery_success_rate": 1.0,  # deterministic movement (env default is 1/3)
            "permute_obs": True,      # per-map random relabeling of observation ids
            "permute_actions": True,  # per-map random relabeling of action ids
        },
    )
    for i in range(NUM_ENVS)
]

env = make_group_env(configs)

## Load model

`load_model` downloads the current checkpoint from the shared Hub repo and reconstructs the full `Model` object with its saved weights. You don't need to re-specify the architecture — it's stored alongside the weights.

Set `FORCE_DOWNLOAD = False` in the cell above to use the local Hub cache instead of re-downloading.

Cached incremental decode uses **FlexAttention** inside `model(..., use_cache=True)`. On CUDA, `model.to(..., dtype=torch.bfloat16)` runs the encoder and backbone in **bfloat16** (the cell above does this automatically) so the flex kernel can compile; **output heads always stay float32** for stable Q-values. The first few calls trigger a one-time warmup (tens of seconds), then per-step decode is much faster than sequential `B=1` forwards. CPU inference stays float32 end-to-end and uses the eager flex path. The encoder and heads stay eager — they process Python dicts and are not the bottleneck.

In [3]:
model = load_model(MODEL_ID, force_download=True, map_location="cpu").eval()
if device.type == "cuda":
    model = model.to(device=device, dtype=torch.bfloat16)  # FlexAttention compile path requires bf16/fp16 on GPU
else:
    model = model.to(device)

config.json:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.76GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

## Batched incremental inference with FlexAttention

At inference time you don't need to re-process the full history on every step. Cached decoding (`use_cache=True`) runs through a **FlexAttention session** that stores per-sequence key/value states, so each new step only processes that row's new tokens.

All envs here advance in lockstep — one new step each per loop iteration — so instead of calling the model once per env, the new steps are stacked into a single `[NUM_ENVS][1]` batch that shares **one FlexAttention decode session** (carried in the model cache). That's one `B=NUM_ENVS` forward per timestep instead of `NUM_ENVS` sequential `B=1` forwards, which is where almost all of the wall-clock time goes.

Pass `use_cache=True` and thread the cache object across steps:

```python
kv_cache = None

with torch.no_grad():
    predictions, _, kv_cache = model(batch, cache=kv_cache, use_cache=True)  # batch: [B][1] step dicts
    actions = model.get_action(predictions, temperature=0.0, num_actions=num_actions)  # [B]
```

`model.get_action` reads the action head's output from the last step in `predictions` for every batch row. Use `temperature=0.0` for greedy argmax; increase it above 0 to sample stochastically.

Rows in a cached batch don't need equal lengths: if an env ever emitted a *variable-sized* response (a different number of steps per call), `model()` decodes each row through an internal FlexAttention session, so every row attends only to its own causal prefix (see `tests/test_kv_cache.py::test_ragged_batched_chunks_match_unbatched`). Lockstep envs like these always contribute one step each per call.

**When to reset the cache:** reset `kv_cache = None` only at **task** boundaries (done code 3 or 4) — the next task is a fresh map with a fresh labeling, so old context is at best useless and at worst misleading. The cache carries across *episode* boundaries within a task. Because all envs here share one batched cache and their task boundaries don't align, the eval loop below keeps per-env row lists and rebuilds the shared cache with one ragged prefill whenever any env crosses a task boundary.

In [4]:
kv_cache = None  # one FlexAttention decode session shared by all env instances
contexts = [[] for _ in env.names]  # rows since each env's current task started
eval_task_scores = [[] for _ in env.names]  # final scores, captured at each task boundary
video_names = env.names[:NUM_VIDEOS]
video_envs = env.envs[:NUM_VIDEOS]
frames_per_env = [[] for _ in video_names]
inputs = None
outputs = None
env.metrics.clear()

# Every env instance uses the same FrozenLake action space, so one num_actions
# applies to the whole batch.
num_actions = env.action_space.spaces[0].n

# Task-budget evaluation: every env gets TASKS_PER_ENV tasks (fresh map each,
# EPISODES_PER_TASK episodes to figure it out), and the score is points per
# task from env.metrics.task_cum_rewards. Truncation at MAX_EPISODE_STEPS
# guarantees the loop finishes. Envs that finish early keep stepping until the
# slowest env is done; scoring below only counts the first TASKS_PER_ENV tasks
# per env.
#
# The model's context must not cross task boundaries (new map, new labeling).
# A shared batched cache can't drop one env's rows, so whenever any env ends
# a task (done code 3/4) its context list is cleared and the cache is rebuilt
# from every env's current-task rows in one ragged batched prefill. An env
# with a just-cleared context takes one random action to start its next task,
# exactly like the first step of the run.
steps_done = 0
while min(len(scores) for scores in eval_task_scores) < TASKS_PER_ENV:
    if outputs is None:
        inputs = env.sample_random_input()
    else:
        new_rows = []
        task_ended = False
        for i, (inp, out) in enumerate(zip(inputs, outputs)):
            row = {**inp, **out}
            row.pop("info", None)
            contexts[i].append(row)
            if int(row["done"]) >= 3:
                contexts[i] = []
                task_ended = True
            new_rows.append(row)
        if task_ended:
            kv_cache = None
            batch = [list(c) for c in contexts]  # ragged rebuild; cleared rows are empty
        else:
            batch = [[row] for row in new_rows]  # lockstep: one new row per env
        with torch.no_grad():
            predictions, _, kv_cache = model(batch, cache=kv_cache, use_cache=True)
            actions = model.get_action(predictions, temperature=0.0, num_actions=num_actions)
        random_inputs = env.sample_random_input()
        inputs = [
            {"action": action} if contexts[i] else random_inputs[i]
            for i, action in enumerate(actions.cpu().numpy())
        ]
    outputs = env.step(inputs)
    for i, out in enumerate(outputs):
        if int(out["done"]) >= 3:
            eval_task_scores[i].append(env.metrics.task_cum_rewards[i][-1])
    for i, video_env in enumerate(video_envs):
        if len(frames_per_env[i]) >= MAX_VIDEO_FRAMES:
            continue
        frames = video_env.render()
        if frames:
            frames_per_env[i].append(frames[-1])
    steps_done += 1
    if steps_done % 100 == 0:
        task_scores = [r for env_tasks in env.metrics.task_cum_rewards for r in env_tasks]
        mean_task_score = sum(task_scores) / len(task_scores) if task_scores else float("nan")
        slowest = min(len(scores) for scores in eval_task_scores)
        print(
            f"step {steps_done} | slowest env {slowest}/{TASKS_PER_ENV} tasks done"
            f" | {len(task_scores)} tasks this window | mean task score {mean_task_score:.2f}/{EPISODES_PER_TASK}"
        )
        env.metrics.clear()  # each progress line covers only the last 100 steps

env.close()

for name, task_scores in zip(env.names, eval_task_scores):
    task_scores = task_scores[:TASKS_PER_ENV]
    scores_text = " ".join(f"{score:.0f}" for score in task_scores)
    print(
        f"{name}: task scores [{scores_text}] / {EPISODES_PER_TASK}"
        f" | mean {sum(task_scores) / TASKS_PER_ENV:.2f}"
    )

step 100 | slowest env 0/4 tasks done | 1 tasks this window | mean task score 0.00/5
step 200 | slowest env 1/4 tasks done | 16 tasks this window | mean task score 1.75/5
step 300 | slowest env 2/4 tasks done | 9 tasks this window | mean task score 0.00/5
step 400 | slowest env 2/4 tasks done | 3 tasks this window | mean task score 0.00/5
step 500 | slowest env 3/4 tasks done | 10 tasks this window | mean task score 0.70/5
proc_frozenlake_0: task scores [0 0 0 0] / 5 | mean 0.00
proc_frozenlake_1: task scores [0 4 0 0] / 5 | mean 1.00
proc_frozenlake_2: task scores [0 2 4 0] / 5 | mean 1.50
proc_frozenlake_3: task scores [0 4 0 0] / 5 | mean 1.00
proc_frozenlake_4: task scores [0 4 5 0] / 5 | mean 2.25
proc_frozenlake_5: task scores [0 0 0 0] / 5 | mean 0.00
proc_frozenlake_6: task scores [0 0 0 5] / 5 | mean 1.25
proc_frozenlake_7: task scores [0 5 0 0] / 5 | mean 1.25
proc_frozenlake_8: task scores [0 0 0 0] / 5 | mean 0.00
proc_frozenlake_9: task scores [0 0 0 0] / 5 | mean 0.00


## Replay MP4s

Frames were collected from the first `NUM_VIDEOS` env instances during the evaluation run above (capped at `MAX_VIDEO_FRAMES` frames each so the replays stay small even when the task budget takes many env steps). Each captured env instance is displayed as its own embedded MP4 so the notebook output stays self-contained without creating separate media files to version.

In [5]:
def display_env_replay(name, frames, *, fps=5):
    print(name)
    h, w = frames[0].shape[:2]
    fig, ax = plt.subplots(figsize=(w / 100, h / 100))
    ax.axis("off")
    img = ax.imshow(frames[0])

    def update(t):
        img.set_data(frames[t])
        return (img,)

    ani = animation.FuncAnimation(
        fig,
        update,
        frames=len(frames),
        interval=1000 / fps,
        blit=True,
    )
    plt.close(fig)
    display(HTML(ani.to_html5_video()))


for name, frames in zip(video_names, frames_per_env):
    display_env_replay(name, frames)

proc_frozenlake_0


proc_frozenlake_1


proc_frozenlake_2


proc_frozenlake_3


proc_frozenlake_4


proc_frozenlake_5


proc_frozenlake_6


proc_frozenlake_7


proc_frozenlake_8


proc_frozenlake_9
